In [2]:
import os
import re
import torch
import datetime
import librosa
import numpy as np
import soundfile as sf
import wave
import weave
from huggingface_hub import snapshot_download, login
from pathlib import Path
from qwen_tts import Qwen3TTSModel
from IPython.display import Audio
from openai import OpenAI
import soundfile as sf
from dotenv import load_dotenv

load_dotenv(override=True)

/bin/sh: sox: command not found
SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


True

In [3]:
HF_TOKEN = os.environ.get('HUGGINGFACEHUB_API_TOKEN')
login(token=HF_TOKEN)

In [4]:
weave.init(f"{os.getenv("WANDB_ACCOUNT")}/{os.getenv("WEAVE_PROJECT")}")

weave: wandb version 0.27.0 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: weave version 0.52.41 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: jbutt.
weave: View Weave data at https://wandb.ai/wandb-smle/jb_qwen_tts/weave


In [ ]:
def get_model_path(model_type: str, model_size: str) -> str:
    """Get model path based on type and size."""
    return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-{model_type}")

def write_sound_to_file(
    output_filename: str,
    audio_file: np.array,
    sample_rate: int,
    ) -> None:

    # Write the file
    # subtype='PCM_16' ensures it's a standard WAV file compatible with all players
    sf.write(
      output_filename,
      audio_file,
      sample_rate,
      subtype='PCM_16',
      )

@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
def generate_voice_design(
  text: str,
  language: str,
  voice_description: str,
  ):
    """Generate speech using Voice Design model (1.7B only)."""

    if not text or not text.strip():
        return None, "Error: Text is required."
    if not voice_description or not voice_description.strip():
        return None, "Error: Voice description is required."

    try:
        wavs, sr = voice_design_model.generate_voice_design(
            text=text.strip(),
            language=language,
            instruct=voice_description.strip(),
            non_streaming_mode=True,
            max_new_tokens=2048,
        )
        print("Voice design generation completed successfully!")

        generated_audio_dir = Path("./audio/designed_audio/")
        generated_audio_dir.mkdir(parents=True, exist_ok=True)
        output_filename = generated_audio_dir / "generated_audio.wav"

        # Write the output to file
        write_sound_to_file(
            output_filename = output_filename,
            audio_file = wavs[0],
            sample_rate = sr,
            )

        print(f"Voice design generation written to file {str(output_filename)}")

        #return (sr, wavs[0])
    except Exception as e:
        return None, f"Error: {type(e).__name__}: {e}"

    # Return the file to Weave op
    return {"audio": wave.open(str(output_filename), "rb"), "sample_rate": sr, "audio_array": wavs[0]}


In [ ]:
# Load the Model
try: 
  print(voice_design_model)
except:
  voice_design_model = Qwen3TTSModel.from_pretrained(
      get_model_path("VoiceDesign", "1.7B"),
      device_map=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps"),
      dtype=torch.bfloat16,
      token=HF_TOKEN,
      #attn_implementation="kernels-community/flash-attn3",
  )

In [ ]:
design_text = "You wouldn't suspect little old me, would you?"
design_language = "Auto"
design_instruct = "Use a playful, seductive and mature female voice. The vocal signature should be female but deeper."

designed_voice_output = generate_voice_design(
    text = design_text,
    language=design_language,
    voice_description=design_instruct,
)

In [ ]:
# Generated Audio
display(
    Audio(
        designed_voice_output["audio_array"],
        rate=designed_voice_output["sample_rate"],
        autoplay=False
        )
    )

# Use Weave Model Class

In [ ]:
from typing import Any
from pydantic import PrivateAttr
from weave import Model

In [15]:
def _drop_audio_array(output):
    """Hide audio_array from the Weave trace; runtime return value is unchanged."""
    if isinstance(output, dict) and "audio_array" in output:
        return {k: v for k, v in output.items() if k != "audio_array"}
    return output


class VoiceDesignModel(Model):
    model_size: str
    model_type: str

    # Lazy-loaded Qwen3TTSModel weights. PrivateAttr → not validated, not serialized
    # into the Weave Model version, not shown as an input on traces.
    _qwen_model: Any = PrivateAttr(default=None)

    def _get_model_path(self, model_type: str, model_size: str) -> str:
        """Get model path based on type and size."""
        return snapshot_download(f"Qwen/Qwen3-TTS-12Hz-{model_size}-{model_type}")

    def _write_sound_to_file(
        self,
        output_filename: str,
        audio_file: np.array,
        sample_rate: int,
        ) -> None:

        # Write the file
        # subtype='PCM_16' ensures it's a standard WAV file compatible with all players
        sf.write(
        output_filename,
        audio_file,
        sample_rate,
        subtype='PCM_16',
        )

    def _ensure_loaded(self):
        """Load Qwen3TTSModel on first use; reuse for the rest of the session."""
        if self._qwen_model is None:
            print(f"Loading Qwen3-TTS-12Hz-{self.model_size}-{self.model_type} (first call only)...")
            
            self._qwen_model = Qwen3TTSModel.from_pretrained(
                self._get_model_path(self.model_type, self.model_size),
                device_map=torch.device("cuda") if torch.cuda.is_available() else torch.device("mps"),
                dtype=torch.bfloat16,
                token=os.environ.get("HUGGINGFACEHUB_API_TOKEN"),
            )
        return self._qwen_model

    #@weave.op(call_display_name=lambda call: f"{call.func_name}-{int(datetime.datetime.now().timestamp())}")
    def generate_voice_design(
        self,
        text: str,
        language: str,
        voice_description: str,
        ):
        """Generate speech using Voice Design model. predict() is the @weave.op entrypoint."""

        if not text or not text.strip():
            return None, "Error: Text is required."
        if not voice_description or not voice_description.strip():
            return None, "Error: Voice description is required."

        try:
            wavs, sr = self._ensure_loaded().generate_voice_design(
                text=text.strip(),
                language=language,
                instruct=voice_description.strip(),
                non_streaming_mode=True,
                max_new_tokens=2048,
            )
            print("Voice design generation completed successfully!")

            generated_audio_dir = Path("./audio/designed_audio/")
            generated_audio_dir.mkdir(parents=True, exist_ok=True)
            output_filename = generated_audio_dir / "generated_audio.wav"

            # Write the output to file
            self._write_sound_to_file(
                output_filename = output_filename,
                audio_file = wavs[0],
                sample_rate = sr,
                )

            print(f"Voice design generation written to file {str(output_filename)}")

        except Exception as e:
            return None, f"Error: {type(e).__name__}: {e}"

        return {"audio": wave.open(str(output_filename), "rb"), "sample_rate": sr, "audio_array": wavs[0]}

    @weave.op(
        call_display_name="VoiceDesign",
        postprocess_output=_drop_audio_array,
    )
    def predict(self, text: str, language: str, voice_description: str):
        return self.generate_voice_design(text, language, voice_description)


In [16]:
voice_design_model = VoiceDesignModel(model_size="1.7B", model_type="VoiceDesign")

In [17]:
design_text = "You wouldn't suspect little old me, would you?"
design_language = "Auto"
design_instruct = "Use a playful, seductive and mature female voice. The vocal signature should be female but deeper."

designed_voice_output = voice_design_model.predict(
    text = design_text,
    language=design_language,
    voice_description=design_instruct,
)

Loading Qwen3-TTS-12Hz-1.7B-VoiceDesign (first call only)...


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

weave: 🍩 https://wandb.ai/wandb-smle/jb_qwen_tts/r/call/019e77fe-cd09-7298-b6f6-dd6ebf948299
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Voice design generation completed successfully!
Voice design generation written to file audio/designed_audio/generated_audio.wav


In [18]:
# Generated Audio
display(
    Audio(
        designed_voice_output["audio_array"],
        rate=designed_voice_output["sample_rate"],
        autoplay=False
        )
    )